# Nougat — OCR matemático para Ukishima
**Instrucciones:**
1. Menú → Entorno de ejecución → Cambiar tipo de entorno → **GPU T4**
2. Ejecuta las celdas en orden
3. En la celda de subida, selecciona tu PDF
4. Al final descarga el `.mmd` y ponlo en `pdfs/salida/` de tu proyecto

In [ ]:
# 1. Instalar Nougat con versión compatible de transformers
!pip install -q nougat-ocr "transformers==4.35.2"

In [ ]:
# 2. Verificar GPU
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Sin GPU — será lento. Cambia el entorno de ejecución a GPU T4.")

In [ ]:
# 3. Subir PDF
from google.colab import files
import pathlib

uploaded = files.upload()
pdf_path = pathlib.Path(list(uploaded.keys())[0])
print(f"Archivo subido: {pdf_path}  ({pdf_path.stat().st_size // 1024} KB)")

In [ ]:
# 4. Procesar con Nougat
import subprocess, os

salida = pathlib.Path("salida")
salida.mkdir(exist_ok=True)

cmd = [
    "nougat",
    str(pdf_path),
    "--out", str(salida),
    "--no-skipping",
]

print("Procesando... (puede tardar varios minutos con GPU, horas sin GPU)")
result = subprocess.run(cmd, text=True)
print("Código de salida:", result.returncode)

In [ ]:
# 5. Buscar el archivo generado
import os

# Busca en toda la carpeta salida/ recursivamente
todos = list(salida.rglob("*"))
print("Archivos generados en salida/:")
for f in todos:
    print(" ", f, f"({f.stat().st_size // 1024} KB)" if f.is_file() else "(carpeta)")

mmd_files = list(salida.rglob("*.mmd"))
if not mmd_files:
    print("\nNo se encontro .mmd — revisa la salida de la celda anterior.")
else:
    mmd = mmd_files[0]
    contenido = mmd.read_text(encoding="utf-8")
    print(f"\nArchivo: {mmd}  ({len(contenido) // 1024} KB)")
    print("\n── Primeras 40 lineas ──")
    print("\n".join(contenido.splitlines()[:40]))

In [ ]:
# 6. Descargar
if mmd_files:
    nombre_final = pdf_path.stem + ".mmd"
    destino = salida / nombre_final
    if mmd_files[0] != destino:
        mmd_files[0].rename(destino)
    files.download(str(destino))
    print(f"Descargando {nombre_final}...")
    print("Ponlo en pdfs/salida/ de tu proyecto Ukishima.")
else:
    print("No hay archivo para descargar.")